# 02 - Model training, tuning and MLflow experiment tracking
Runs `python -m heartrisk.train` (config: `configs/train.yaml`): Logistic Regression, Random Forest and Gradient Boosting, each tuned with GridSearchCV + stratified 5-fold CV; every hyper-parameter combination is logged as an MLflow child run. Start the UI with `mlflow ui --backend-store-uri sqlite:///mlflow.db`.

In [ ]:
# Make the repository's src/ importable whether or not `pip install -e .` was run
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import os; os.chdir(ROOT)
import platform, socket, datetime
print("Run on host:", socket.gethostname(), "| Python", platform.python_version(),
      "| started", datetime.datetime.now().isoformat(timespec="seconds"))

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s', force=True)
from heartrisk.train import run
out = run()   # same code path as `python -m heartrisk.train`
print('Selected:', out['selected_model'])

In [ ]:
import json, pandas as pd
m = json.load(open('reports/metrics.json'))
cols = ['model','cv_roc_auc_mean','cv_roc_auc_std','cv_accuracy_mean','cv_precision_mean','cv_recall_mean','cv_f1_mean',
        'test_roc_auc','test_accuracy','test_precision','test_recall','overfit_gap_roc_auc']
pd.DataFrame(m['models'])[cols].round(4)

In [ ]:
print('Selected model:', m['selected_model'])
print('Reason:', m['selection_reason'])
for r in m['models']:
    print(r['model'], 'best params ->', r['best_params'])

In [ ]:
from IPython.display import Image, display
display(Image('reports/figures/models/model_comparison.png'))
sel = m['selected_model']
for f in ['roc_curve','confusion_matrix','feature_importance']:
    display(Image(f'reports/figures/models/{sel}/{f}.png'))

## Runs recorded in MLflow

In [ ]:
import mlflow
mlflow.set_tracking_uri('sqlite:///mlflow.db')
runs = mlflow.search_runs(experiment_names=['heart-disease-uci'])
print(runs['tags.run_level'].value_counts())
runs[runs['tags.run_level']=='parent'][['tags.mlflow.runName','metrics.cv_roc_auc_mean','metrics.test_roc_auc','start_time']]